# PortWatch AI — ARIMA Training Notebook (Local Version)

**Converted from:** Azure Databricks (Spark + ADLS)  
**Target:** Local Python + Pandas + statsmodels  

### Changes from original (22 cells → 9 clean cells):
- **REMOVED:** All `abfss://` paths → local `data/` and `outputs/` paths
- **REMOVED:** `spark.read.parquet(...)` / `spark.read.csv(...)` → `pd.read_parquet()`
- **REMOVED:** `dbutils.fs.ls(...)` (3 exploratory cells)
- **REMOVED:** `spark.createDataFrame(...)` + `.write.parquet()` → `df.to_parquet()`
- **REMOVED:** Duplicate data-loading cells (cells 2-7 were trial-and-error)
- **ADAPTED:** Original read raw CSV using `portcalls` column → now reads
  `data/port_daily_expanded.parquet` using `daily_port_calls` (equivalent data,
  already cleaned by upstream notebooks)
- **PRESERVED:** All ARIMA modeling logic exactly as original

### Input:
- `data/port_daily_expanded.parquet` (from ML setup notebook)

### Outputs:
- `models/arima_model.pkl` — fitted ARIMA model for the selected port
- `outputs/arima_predictions.parquet` — ARIMA predictions (actual vs predicted)

In [1]:
# =============================================================================
# Cell 1 — CONFIG & IMPORTS
# =============================================================================
# REMOVED: dbutils.fs.ls(...) exploratory cells (3 cells)
# REMOVED: Multiple trial-and-error data read attempts (cells 2-7)
# =============================================================================

import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error

# --- Local path config ---
BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "data"
MODELS_DIR = BASE_DIR / "models"
OUTPUTS_DIR = BASE_DIR / "outputs"

# Input (from ML setup notebook)
INPUT_PATH = DATA_DIR / "port_daily_expanded.parquet"

# Outputs
ARIMA_MODEL_PATH = MODELS_DIR / "arima_model.pkl"
ARIMA_PREDS_PATH = OUTPUTS_DIR / "arima_predictions.parquet"

MODELS_DIR.mkdir(exist_ok=True)
OUTPUTS_DIR.mkdir(exist_ok=True)

print("Libraries imported successfully")
print(f"Input:  {INPUT_PATH}")
print(f"Model:  {ARIMA_MODEL_PATH}")
print(f"Output: {ARIMA_PREDS_PATH}")

Libraries imported successfully
Input:  data\port_daily_expanded.parquet
Model:  models\arima_model.pkl
Output: outputs\arima_predictions.parquet


In [2]:
# =============================================================================
# Cell 2 — LOAD DATA
# =============================================================================
# REPLACED: spark.read.option("header","true").option("inferSchema","true")
#           .csv("abfss://raw@.../Daily_Port_Activity_Data_and_Trade_Estimates.csv")
#           .toPandas()
#           → pd.read_parquet() from local expanded features
# ADAPTED:  Original used raw CSV with 'portcalls' column.
#           We use port_daily_expanded.parquet with 'daily_port_calls' column
#           (same data, already aggregated to daily level by ingestion notebook).
# =============================================================================

assert INPUT_PATH.exists(), f"ERROR: {INPUT_PATH} not found. Run the ML setup notebook first."

df = pd.read_parquet(INPUT_PATH)

print("Data loaded successfully")
print("Rows:", len(df))
print("Columns:", list(df.columns))

Data loaded successfully
Rows: 125500
Columns: ['event_date', 'portid', 'daily_port_calls', 'year', 'month', 'lag_1', 'lag_2', 'lag_3', 'lag_7', 'lag_14', 'lag_30', 'roll_mean_7', 'roll_mean_14', 'roll_mean_30', 'roll_std_7', 'roll_std_14', 'roll_std_30', 'lag_1_is_null', 'lag_2_is_null', 'lag_3_is_null', 'lag_7_is_null', 'lag_14_is_null', 'lag_30_is_null', 'roll_mean_7_is_null', 'roll_mean_14_is_null', 'roll_mean_30_is_null', 'roll_std_7_is_null', 'roll_std_14_is_null', 'roll_std_30_is_null', 'dow', 'is_weekend', 'port_mean_prev', 'port_count_prev']


In [3]:
# =============================================================================
# Cell 3 — SELECT PORT
# =============================================================================
# PRESERVED: Same logic — pick the port with most data rows
# ADAPTED:  Original used df["portname"].value_counts().index[0]
#           Our data has 'portid' (not 'portname'), so we use portid instead.
#           The logic is identical: pick the top port by frequency.
# =============================================================================

# Pick the port with the most data (same logic as original)
PORT_ID = df["portid"].value_counts().index[0]
print("Using port:", PORT_ID)

# Filter to this port's daily_port_calls time series
# Original: port_df = df[df["portname"] == PORT_NAME][["date", "portcalls"]].copy()
# Adapted:  uses 'event_date' and 'daily_port_calls' from our pipeline
port_df = df[df["portid"] == PORT_ID][["event_date", "daily_port_calls"]].copy()

print("Rows for this port:", len(port_df))

Using port: port1065
Rows for this port: 2510


In [4]:
# =============================================================================
# Cell 4 — PREPARE TIME SERIES
# =============================================================================
# PRESERVED: Exact same logic as original cells 10-11
# ADAPTED:  Column names (event_date → date, daily_port_calls → portcalls)
# =============================================================================

# Convert date to datetime (original: port_df["date"] = pd.to_datetime(port_df["date"]))
port_df["event_date"] = pd.to_datetime(port_df["event_date"])

# Sort by time
port_df = port_df.sort_values("event_date")

# Remove missing values
port_df = port_df.dropna()

print("Head:")
print(port_df.head().to_string())
print("\nTail:")
print(port_df.tail().to_string())

# Create time series (original: ts = port_df.set_index("date")["portcalls"])
ts = port_df.set_index("event_date")["daily_port_calls"]
print(f"\nTime series length: {len(ts)}")
print(f"Date range: {ts.index.min()} → {ts.index.max()}")

Head:
  event_date  daily_port_calls
0 2019-01-01                57
1 2019-01-02                53
2 2019-01-03                51
3 2019-01-04                50
4 2019-01-05                52

Tail:
     event_date  daily_port_calls
2505 2025-11-10                48
2506 2025-11-11                50
2507 2025-11-12                46
2508 2025-11-13                47
2509 2025-11-14                46

Time series length: 2510
Date range: 2019-01-01 00:00:00 → 2025-11-14 00:00:00


In [5]:
# =============================================================================
# Cell 5 — TRAIN/TEST SPLIT
# =============================================================================
# PRESERVED: Exact same logic as original cell 12
#            80/20 temporal split — no data leakage
# =============================================================================

split_idx = int(len(ts) * 0.8)

train = ts.iloc[:split_idx]
test = ts.iloc[split_idx:]

print("Train size:", len(train))
print("Test size:", len(test))
print(f"Train period: {train.index.min()} → {train.index.max()}")
print(f"Test period:  {test.index.min()} → {test.index.max()}")

Train size: 2008
Test size: 502
Train period: 2019-01-01 00:00:00 → 2024-06-30 00:00:00
Test period:  2024-07-01 00:00:00 → 2025-11-14 00:00:00


In [6]:
# =============================================================================
# Cell 6 — FIT ARIMA MODEL
# =============================================================================
# PRESERVED: Exact same logic as original cell 14
#            ARIMA(1, 1, 1) — same order parameters
# =============================================================================

print("Fitting ARIMA(1,1,1) model...")
model = ARIMA(train, order=(1, 1, 1))
model_fit = model.fit()

print(model_fit.summary())

Fitting ARIMA(1,1,1) model...


D:\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
D:\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
D:\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


                               SARIMAX Results                                
Dep. Variable:       daily_port_calls   No. Observations:                 2008
Model:                 ARIMA(1, 1, 1)   Log Likelihood               -7401.882
Date:                Wed, 25 Mar 2026   AIC                          14809.765
Time:                        17:57:35   BIC                          14826.578
Sample:                    01-01-2019   HQIC                         14815.937
                         - 06-30-2024                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.1163      0.014      8.382      0.000       0.089       0.144
ma.L1         -0.9737      0.005   -201.472      0.000      -0.983      -0.964
sigma2        93.3925      1.491     62.631      0.0

In [7]:
# =============================================================================
# Cell 7 — FORECAST & EVALUATE
# =============================================================================
# PRESERVED: Exact same logic as original cells 16-17
# ADAPTED:  Column names to match our pipeline
# =============================================================================

# Forecast (original: preds = model_fit.forecast(steps=len(test)))
preds = model_fit.forecast(steps=len(test))
mae = mean_absolute_error(test, preds)

print(f"ARIMA MAE: {mae:.4f}")

# Align predictions with test dates (original cell 17 logic preserved exactly)
arima_df = pd.DataFrame({
    "date": test.index,
    "portid": PORT_ID,
    "actual_portcalls": test.values,
    "predicted_portcalls": preds.values,
    "model": "ARIMA"
})

# Reset index (original: arima_df = arima_df.reset_index(drop=True))
arima_df = arima_df.reset_index(drop=True)

print("\nPredictions preview:")
print(arima_df.head(10).to_string())

ARIMA MAE: 6.5166

Predictions preview:
        date    portid  actual_portcalls  predicted_portcalls  model
0 2024-07-01  port1065                68            53.209359  ARIMA
1 2024-07-02  port1065                50            54.047893  ARIMA
2 2024-07-03  port1065                71            54.145425  ARIMA
3 2024-07-04  port1065                54            54.156769  ARIMA
4 2024-07-05  port1065                53            54.158088  ARIMA
5 2024-07-06  port1065                59            54.158241  ARIMA
6 2024-07-07  port1065                42            54.158259  ARIMA
7 2024-07-08  port1065                48            54.158261  ARIMA
8 2024-07-09  port1065                60            54.158262  ARIMA
9 2024-07-10  port1065                62            54.158262  ARIMA


In [8]:
# =============================================================================
# Cell 8 — SAVE MODEL & PREDICTIONS
# =============================================================================
# REPLACED: spark.createDataFrame(arima_df).write.mode("overwrite").parquet(
#           "abfss://predictions@.../arima/daily_port_calls/")
#           → df.to_parquet(local path)
# REMOVED: spark.createDataFrame(...) conversion
# REMOVED: spark_arima_df.printSchema()
# ADDED:   joblib.dump() to save the fitted ARIMA model for reuse
# =============================================================================

# Save predictions
print(f"Writing ARIMA predictions ({len(arima_df)} rows) to: {ARIMA_PREDS_PATH}")
arima_df.to_parquet(ARIMA_PREDS_PATH, index=False)
print(f"✓ Predictions saved ({ARIMA_PREDS_PATH.stat().st_size:,} bytes)")

# Save model
print(f"\nSaving ARIMA model to: {ARIMA_MODEL_PATH}")
joblib.dump(model_fit, ARIMA_MODEL_PATH)
print(f"✓ Model saved ({ARIMA_MODEL_PATH.stat().st_size:,} bytes)")

Writing ARIMA predictions (502 rows) to: outputs\arima_predictions.parquet
✓ Predictions saved (8,388 bytes)

Saving ARIMA model to: models\arima_model.pkl
✓ Model saved (5,866,939 bytes)


In [9]:
# =============================================================================
# Cell 9 — VALIDATE
# =============================================================================

# Verify readback
val = pd.read_parquet(ARIMA_PREDS_PATH)
print(f"Readback: {len(val)} rows")
print(f"Columns: {list(val.columns)}")
print(val.head().to_string())

# Verify model reload
loaded = joblib.load(ARIMA_MODEL_PATH)
print(f"\n✓ Model reload verified: {type(loaded)}")

# Summary
print(f"\n{'='*60}")
print(f"NOTEBOOK COMPLETE")
print(f"{'='*60}")
print(f"  Port modeled: {PORT_ID}")
print(f"  ARIMA order:  (1, 1, 1)")
print(f"  MAE:          {mae:.4f}")
print(f"  Train size:   {len(train)}")
print(f"  Test size:    {len(test)}")
print(f"\nOutput files:")
print(f"  1. {ARIMA_PREDS_PATH}")
print(f"  2. {ARIMA_MODEL_PATH}")

Readback: 502 rows
Columns: ['date', 'portid', 'actual_portcalls', 'predicted_portcalls', 'model']
        date    portid  actual_portcalls  predicted_portcalls  model
0 2024-07-01  port1065                68            53.209359  ARIMA
1 2024-07-02  port1065                50            54.047893  ARIMA
2 2024-07-03  port1065                71            54.145425  ARIMA
3 2024-07-04  port1065                54            54.156769  ARIMA
4 2024-07-05  port1065                53            54.158088  ARIMA

✓ Model reload verified: <class 'statsmodels.tsa.arima.model.ARIMAResultsWrapper'>

NOTEBOOK COMPLETE
  Port modeled: port1065
  ARIMA order:  (1, 1, 1)
  MAE:          6.5166
  Train size:   2008
  Test size:    502

Output files:
  1. outputs\arima_predictions.parquet
  2. models\arima_model.pkl
